# Монгол хэл дээр LLaMA 3 загвар сургах заавар

"Энэхүү ноутбук нь Google Colab дээр LLaMA 3 загварыг монгол хэл дээр сургах, дараа нь ашиглах явцыг харуулна. Та Colab ашиглаж, runtime төрлийг GPU-тай болгож өөрчлөх хэрэгтэй.

## Ажлын дараалал:

1. Шаардлагатай сангуудыг суулгах,
2. Hugging Face токеноор нэвтрэх,
3. LLaMA 3 загварыг ачаалах,
4. Монгол хэлний өгөгдөл ачаалах,
5. Өгөгдлийг боловсруулах, токенжуулах,
6. Загварыг сургах,
7. Сургасан загварыг хадгалах,
8. Загварыг туршиж үзэх,
9. Загваыг асуулт хариултын сангаар сургах

In [1]:
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import torch.nn.functional as F
import numpy as np
import random
import os
import json
import argparse
import time
import logging
import re
from tqdm import tqdm
from typing import List, Dict, Any, Tuple
from datasets import load_dataset

In [2]:
import huggingface_hub
huggingface_hub.login(token="")

In [3]:
# !pip install --upgrade transformers

In [4]:
# !pip install bitsandbytes

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

import bitsandbytes as bnb # int8
quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0, # Optional: Adjust the outlier threshold if needed
    # Optional: other parameters like llm_int8_skip_modules for specific layers
)

import torch
model_name = "Qwen/Qwen3.5-4B"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto", # Automatically distributes the model across available GPUs
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
# Essential for Llama models
tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

In [6]:
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM

# create LoRA configuration object
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, # type of task to train on
    inference_mode=False, # set to False for training
    r=8, # dimension of the smaller matrices
    lora_alpha=32, # scaling factor
    lora_dropout=0.1, # dropout of LoRA layers
    target_modules=['q_proj', 'k_proj', 'v_proj'] # Specify target modules
)

# Wrap the base model with PEFT configuration
model = get_peft_model(model, lora_config)

In [7]:
# dataset.push_to_hub("orgilj/cleaned_pretrain_dataset")

In [8]:
from datasets import load_dataset
dataset = load_dataset("saillab/alpaca-mongolian-cleaned")
train_test_dataset = dataset['train'].select(range(10000)).train_test_split(test_size=0.05, seed=42) # 10000 samples for training and 500 for testing

In [9]:
def tokenize_function(examples):
    # Set a consistent max length that works for your data
    max_length = 2048  # Adjust as needed
    # examples['instruction']= examples['instruction']+examples['input']
    return tokenizer(
        examples["instruction"],
        truncation=True,
        padding="max_length",
        max_length=max_length,
        return_tensors=None,  # Important - don't return tensors yet
    )

tokenized_dataset = train_test_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["instruction", 'input'],
    num_proc=1 # Added to resolve potential ArrowInvalid due to multiprocessing inconsistency
)

In [10]:
tokenizer.pad_token = tokenizer.eos_token

In [11]:
from transformers import DataCollatorForLanguageModeling, TrainingArguments, Trainer

# The data collator will handle padding dynamically
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # We're doing causal language modeling, not masked LM
)

training_args = TrainingArguments(
    output_dir="./qwen35-mongolian",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    eval_strategy="steps",
    eval_steps=500,
    logging_dir="./logs",
    logging_steps=100,
    save_steps=500,
    save_total_limit=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=False,  # Keep as False when bf16 is True
    bf16=True,   # Enable BFloat16 mixed precision for memory efficiency
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    warmup_steps=500,
    report_to="tensorboard",
    gradient_checkpointing=True, # Enable gradient checkpointing to save memory
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss,Validation Loss


### Дүгнэлт

Монгол хэлний талаар тодорхой хэмжээний ойлголттой болсон бөгөөд үүнийг бүх өгөглийн санд багц багцаар сургалт хийж сайжруулж болох юм.

Одоо бид сургасан загвараа хадгалж түүнийг асуултанд хэрхэн хариулж байгааг туршиж үзье

In [ ]:
# Save model and tokenizer
model_save_path = "./llama3-mongolian-finetuned"
trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load your fine-tuned model and tokenizer
model_path = "./llama3-mongolian-finetuned"  # Path where you saved the model
model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.float16, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Set pad token if needed
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Prepare your prompt
prompt = "Сайн байна уу? Миний нэрийг Монгол гэдэг"  # Example Mongolian prompt

# Tokenize the prompt
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Generate text
outputs = model.generate(
    inputs.input_ids,
    max_length=100,                # Maximum length of generated text
    temperature=0.7,               # Controls randomness (lower = more deterministic)
    top_p=0.9,                     # Nucleus sampling parameter
    do_sample=True,                # Use sampling instead of greedy decoding
    no_repeat_ngram_size=2,        # Avoid repeating 2-grams
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

# Decode the generated text
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(generated_text)

In [ ]:
# Create a chat message
messages = [
    {"role": "user", "content": "Монгол хэл дээр надад мэндчилгээ бичиж өгөөч."}
]

# Format using chat template if available
if hasattr(tokenizer, "apply_chat_template"):
    chat_input = tokenizer.apply_chat_template(messages, return_tensors="pt").to(model.device)
else:
    # Fallback if no chat template
    chat_input = tokenizer(messages[0]["content"], return_tensors="pt").to(model.device)

# Generate the response
outputs = model.generate(
    chat_input,
    max_new_tokens=200,            # Only generate this many new tokens
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

# Decode and print the response
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(generated_text)

In [ ]:
prompts = [
    "Монгол улсын нийслэл хот аль вэ?",
    "Монголын алдартай хоол юу вэ?",
    "Монгол хэл дээр шүлэг зохиогоорой"
]

# Tokenize and create batch
batch_inputs = tokenizer(prompts, padding=True, return_tensors="pt").to(model.device)

# Generate text for all prompts
batch_outputs = model.generate(
    batch_inputs.input_ids,
    max_length=150,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

# Decode all generated texts
for i, output in enumerate(batch_outputs):
    generated_text = tokenizer.decode(output, skip_special_tokens=True)
    print(f"Prompt {i+1}: {prompts[i]}")
    print(f"Response: {generated_text}\n")

## Асуулт хариулт туршилт үр дүн

Сургасан загвар асуултанд хариулах чадваргүй бөгөөд бид үүнийг дараах сургалтын санг ашиглаж тус чадварыг суулгаж болох юм.

In [ ]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

# Тохиргооны параметрүүд - 80GB GPU-д тохируулсан
MODEL_NAME = "./llama3-mongolian-finetuned"  # Таны загварын зам
DATASET_NAME = "margiela00/general_qa"  # Асуулт-хариултын өгөгдлийн сангийн нэр
OUTPUT_DIR = "./llama32_3b_qa_model"  # Гаралтын директори
BATCH_SIZE = 4  # Batch хэмжээг бууруулах
GRADIENT_ACCUMULATION_STEPS = 8  # Градиент хуримтлуулах алхмын тоо
LEARNING_RATE = 1e-5  # Суралцах хурд (бүтэн сургалтад доогуур түвшин ашиглана)
NUM_EPOCHS = 2  # Эргэлтийн тоо
MAX_LENGTH = 512  # Текстийн максимум урт
WEIGHT_DECAY = 0.01  # Жин унах параметр
WARMUP_RATIO = 0.03  # Дулаацах харьцаа

# TOKENIZERS_PARALLELISM сэрэмжлүүлгийг хаах
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# GPU тохиргоо
device_map = "auto"
world_size = int(os.environ.get("WORLD_SIZE", 1))
ddp = world_size != 1

# Загвар болон токенайзер ачаалах
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

# Загварыг бүтэн нарийвчлалтай ачаалах - bf16 ашиглах (fp16 биш)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,  # fp16 биш bfloat16 ашиглах
    device_map=device_map,
)

# Өгөгдлийн сан ачаалах
dataset = load_dataset(DATASET_NAME)

# Өгөгдлийн талаар мэдээлэл харуулах
print(f"Өгөгдлийн сангийн бүтэц: {dataset}")
print(f"Сургалтын өгөгдлийн баганууд: {dataset['train'].column_names}")
print(f"Жишээ бичлэг: {dataset['train'][0]}")

# Өгөгдлийг форматлах функц - 'instruction', 'input', 'output' бүтцийг харгалзах
def format_qa_prompt(example):
    """Асуулт хариултын өгөгдлийг загварт тохирох форматад хөрвүүлэх"""
    # Хэрэв input утга None биш бол нэмж оруулах
    if example['input'] and example['input'] != None and example['input'] != "":
        prompt = f"<|user|>\n{example['instruction']}\n{example['input']}\n<|assistant|>\n{example['output']}\n<|endoftext|>"
    else:
        prompt = f"<|user|>\n{example['instruction']}\n<|assistant|>\n{example['output']}\n<|endoftext|>"
    return {"text": prompt}

# Өгөгдлийг форматлах
formatted_dataset = dataset.map(format_qa_prompt)

# Токенжуулах функц - Хэлний загварын хувьд label-ийг автоматаар тохируулах
def tokenize_function(examples):
    outputs = tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        return_special_tokens_mask=True
    )
    return outputs

# Өгөгдлийг токенжуулах
tokenized_dataset = formatted_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=formatted_dataset["train"].column_names,
    num_proc=1  # Parallel процесс ашиглахгүй - deadlock алдааг зайлсхийхийн тулд
)

# Өгөгдөл цуглуулагч - DataCollatorForLanguageModeling ашиглах
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # MLM биш, авторегрессив сургалт
)

# Сургалтын аргументууд тохируулах
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    bf16=True,  # fp16 биш bf16 ашиглах
    fp16=False,  # fp16-г хаах
    remove_unused_columns=False,
    ddp_find_unused_parameters=False if ddp else None,
    gradient_checkpointing=True,  # Градиент шалгах цэгүүдийг идэвхжүүлэх
    report_to="tensorboard",  # TensorBoard ашиглан сургалтыг хянах
    dataloader_num_workers=0,  # Parallel процесс ашиглахгүй - deadlock алдааг зайлсхийхийн тулд
    max_grad_norm=1.0,  # Градиент нормыг хязгаарлах
)

# Trainer объект үүсгэх
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    data_collator=data_collator,
)

# Загварыг сургах
trainer.train()

# Сургасан загварыг хадгалах
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Загварыг дахин ачаалж, туршиж үзэх функц
def generate_response(question, max_length=512):
    model_for_inference = AutoModelForCausalLM.from_pretrained(
        OUTPUT_DIR,
        device_map="auto",
        torch_dtype=torch.bfloat16,  # bfloat16 ашиглах
    )
    prompt = f"<|user|>\n{question}\n<|assistant|>\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model_for_inference.device)

    with torch.no_grad():
        output = model_for_inference.generate(
            **inputs,
            max_length=max_length,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )

    response = tokenizer.decode(output[0], skip_special_tokens=True)
    # <|assistant|> хэсгээс хариултыг гаргаж авах
    assistant_part = response.split("<|assistant|>\n")
    if len(assistant_part) > 1:
        return assistant_part[1]
    else:
        return response

# Туршилт
if __name__ == "__main__":
    question = "Монгол улс хэдэн аймагтай вэ?"
    print(generate_response(question))

# Даалгавар

1. 2 өгөгдлийн санг бүтэн сургалтын багцаар сургах.
2. Сургасан загваруудын хамгийн сайн үр дүн үзүүлсэн загварыг хадгалах.
3. Асуулт хариултын санг хянах нэмэлт асуулт хариулт оруулж сургах.
4. Бусад чадварыг сургах. Функц дуудаж, математикийн асуулт, орчуулга Англи-Монгол гэх мэт аль нэгийг сонгоно.
5. Бусад чадварыг сургахдаа DPO, PPO, ORPO аргуудын аль нэгийг ашиглах. (OPTIONAL)
6. CoT Chain of Tought сургах буюу дараалсан бодлуудыг гинж сургах DRPO. (OPTIONAL)

In [ ]:
import os
from huggingface_hub import login
import torch
from datasets import Dataset, load_dataset, concatenate_datasets
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)


login(token="")
print("Hugging Face-д нэвтрэх оролдлого хийлээ")

# model_name = "meta-llama/Llama-3.2-3B-Instruct"  # Мета LLaMA 3 загвар
model_name = "facebook/opt-350m"  # Нээлттэй загвар (Meta-гийн загвар ачаалж чадахгүй бол ашиглана)

print(f"Загварыг ачаалж байна: {model_name}")

# Токенайзер ачаалах
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Загвар ачаалах
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32,  # FP16 биш FP32 ашиглах
    device_map="auto",
)


# Ерөнхий монгол хэлний өгөгдөл
general_mongolian_text = [
    "Монгол улс нь Ази тивд оршдог улс юм. Улаанбаатар нь Монгол улсын нийслэл.",
    "Чингис хаан Монгол эзэнт гүрнийг үүсгэсэн. Тэрээр дэлхийн хамгийн том эзэнт гүрнийг байгуулсан.",
    "Монгол хэл нь Алтайн хэлний бүлэгт хамаарна. Монгол хэлээр ойролцоогоор 5 сая хүн ярьдаг.",
    "Монголын говь цөл нь дэлхийн томоохон цөлүүдийн нэг юм. Говь цөлд хос бөхт тэмээ амьдардаг.",
    "Монгол улсын эдийн засгийн гол салбарууд нь уул уурхай, мал аж ахуй, аялал жуулчлал юм.",
] * 50  # 250 бичлэг болгох

# Асуулт-хариулт өгөгдөл
qa_examples = [
    {"instruction": "Монгол улс хэдэн аймагтай вэ?", "input": "", "output": "Монгол улс нь 21 аймагтай."},
    {"instruction": "Монголын хамгийн өндөр уул юу вэ?", "input": "", "output": "Монголын хамгийн өндөр уул нь Таван Богд."},
    {"instruction": "Монгол хэлний цагаан толгой хэдэн үсэгтэй вэ?", "input": "", "output": "Монгол цагаан толгой нь 35 үсэгтэй."},
    {"instruction": "Монголын үндэсний баяр юу вэ?", "input": "", "output": "Наадам нь Монголын үндэсний баяр юм."},
    {"instruction": "Монголын нийслэл хот юу вэ?", "input": "", "output": "Улаанбаатар нь Монгол улсын нийслэл."},
] * 10  # 50 бичлэг болгох

# ӨГӨГДЛИЙН САНГУУДЫГ ҮҮСГЭХ
general_dataset = Dataset.from_dict({"text": general_mongolian_text})
print(f"Ерөнхий өгөгдлийн сангийн хэмжээ: {len(general_dataset)} бичлэг")

qa_dataset = Dataset.from_dict({
    "instruction": [ex["instruction"] for ex in qa_examples],
    "input": [ex["input"] for ex in qa_examples],
    "output": [ex["output"] for ex in qa_examples]
})
print(f"Асуулт-хариултын өгөгдлийн сангийн хэмжээ: {len(qa_dataset)} бичлэг")

# 4. ӨГӨГДЛИЙГ БОЛОВСРУУЛАХ
# Асуулт-хариултыг боловсруулах
def prepare_qa_dataset(examples):
    texts = []
    for instruction, inp, output in zip(examples["instruction"], examples["input"], examples["output"]):
        if inp:
            text = f"<|user|>\n{instruction}\n{inp}\n<|assistant|>\n{output}\n<|endoftext|>"
        else:
            text = f"<|user|>\n{instruction}\n<|assistant|>\n{output}\n<|endoftext|>"
        texts.append(text)
    return {"text": texts}

# Өгөгдлийг боловсруулах
processed_qa = qa_dataset.map(
    prepare_qa_dataset,
    batched=True,
    remove_columns=qa_dataset.column_names,
)

# Өгөгдлийн сангуудыг нэгтгэх
combined_dataset = concatenate_datasets([general_dataset, processed_qa])
print(f"Нэгтгэсэн өгөгдлийн сангийн хэмжээ: {len(combined_dataset)} бичлэг")

# Сургалт ба тестийн өгөгдөлд хуваах
train_test_dataset = combined_dataset.train_test_split(test_size=0.05, seed=42)
print(f"Сургалтын өгөгдөл: {len(train_test_dataset['train'])} бичлэг")
print(f"Тестийн өгөгдөл: {len(train_test_dataset['test'])} бичлэг")

# 5. ТОКЕНЖУУЛАХ
# Токенжуулах функц
def tokenize_function(examples):
    max_length = 128  # Хэмжээг багасгав

    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=max_length,
        return_tensors=None,
    )

# Өгөгдлийг токенжуулах
print("Өгөгдлийг токенжуулж байна...")
tokenized_dataset = train_test_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

# 6. СУРГАЛТЫН ТОХИРГОО - FP16 биш
# Өгөгдөл цуглуулагч
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Сургалтын аргументууд (fp16=False болгосон!)
training_args = TrainingArguments(
    output_dir="./mongolian-model",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    logging_dir="./logs",
    logging_steps=10,
    save_steps=50,
    save_total_limit=1,
    learning_rate=5e-5,
    weight_decay=0.01,
    fp16=False,  # fp16 идэвхгүй болгосон!
    bf16=False,  # bf16 мөн идэвхгүй
    gradient_accumulation_steps=4,
    num_train_epochs=10,
    warmup_steps=50,
    report_to="tensorboard",
)

# 7. TRAINER ОБЪЕКТ ҮҮСГЭХ
print("Trainer объект үүсгэж байна...")
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
)

# 8. СУРГАЛТЫГ ЭХЛҮҮЛЭХ
print("\nСургалтыг эхлүүлэхэд бэлэн байна")
print("Дараах мөрийг ажиллуулан сургалтыг эхлүүлнэ:")
print("trainer.train()")

# 9. ЗАГВАРЫГ ТЕСТЛЭХ ФУНКЦ
def test_model(prompt, max_length=100):
    """Загварыг туршиж үзэх функц"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        inputs.input_ids,
        max_length=max_length,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        no_repeat_ngram_size=2,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Сургалт дууссаны дараа дараах функцийг ашиглан туршиж үзэх боломжтой:")
print('test_response = test_model("Монгол улс нь ")')
print('print(test_response)')

In [ ]:
trainer.train()

In [ ]:
model_save_path = "./llama3-mongolian-combined"
trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Хадгалсан загварын замыг тодорхойлох
model_save_path = "./llama3-mongolian-combined"

# 1. ЗАГВАР БОЛОН ТОКЕНАЙЗЕР АЧААЛАХ
print(f"Сургасан загварыг ачаалж байна: {model_save_path}")
try:
    # Загварыг ачаалах
    model = AutoModelForCausalLM.from_pretrained(
        model_save_path,
        torch_dtype=torch.float32,
        device_map="auto",
    )

    # Токенайзерыг ачаалах
    tokenizer = AutoTokenizer.from_pretrained(model_save_path)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print("Загвар болон токенайзер амжилттай ачаалагдлаа!")
except Exception as e:
    print(f"Алдаа гарлаа: {e}")
    print("Хэрэв загвар хадгалагдаагүй бол ахин сургалтыг хийх хэрэгтэй.")

# 2. ЗАГВАРЫГ ТУРШИЖ ҮЗЭХ ФУНКЦ
def test_model(prompt, max_length=100):
    """Загварыг туршиж үзэх функц"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        inputs.input_ids,
        max_length=max_length,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        no_repeat_ngram_size=2,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# 3. ЗАГВАРЫГ ҮНЭЛЭХ ТЕСТҮҮД
print("\nЗагварыг үнэлж байна...")

# Ерөнхий текст үүсгэх тест
print("\n--- ЕРӨНХИЙ ТЕКСТ ҮҮСГЭХ ТЕСТ ---")
general_prompts = [
    "Монгол улс нь",
    "Чингис хаан бол",
    "Монголын говь цөл нь"
]

for prompt in general_prompts:
    print(f"\nПромпт: {prompt}")
    response = test_model(prompt, max_length=150)
    print(f"Хариулт: {response}")

# Асуулт-хариултын тест
print("\n--- АСУУЛТ-ХАРИУЛТЫН ТЕСТ ---")
qa_prompts = [
    "<|user|>\nМонгол улс хэдэн аймагтай вэ?\n<|assistant|>",
    "<|user|>\nМонголын хамгийн өндөр уул юу вэ?\n<|assistant|>",
    "<|user|>\nМонгол хэлний цагаан толгой хэдэн үсэгтэй вэ?\n<|assistant|>"
]

for prompt in qa_prompts:
    print(f"\nПромпт: {prompt}")
    response = test_model(prompt, max_length=150)
    print(f"Хариулт: {response}")

# 4. ҮР ДҮНГ НЭГТГЭХ
print("\n--- ЗАГВАРЫН ҮНЭЛГЭЭНИЙ ДҮГНЭЛТ ---")
print("Сургасан загварын гүйцэтгэлийг шалгаж дууслаа.")
print("Амжилттай хадгалагдсан загварыг дараагийн даалгаварт ашиглах боломжтой байна.")
print("Загварын чанарыг сайжруулахын тулд дараагийн даалгавар руу шилжих боломжтой.")

# 5. ДАРААГИЙН АЛХАМ
print("\n--- ДАРААГИЙН АЛХАМ ---")
print("3-р алхам: Асуулт хариултын санг хянах нэмэлт асуулт хариулт оруулж сургах")

In [ ]:
import os
from huggingface_hub import login
import torch
from datasets import Dataset, load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

# 1. HUGGING FACE-Д НЭВТРЭХ (ХЭРЭВ ШААРДЛАГАТАЙ)
try:
    login(token="")
    print("Hugging Face-д нэвтэрлээ")
except Exception as e:
    print(f"Нэвтрэх үед алдаа гарлаа: {e}")
    print("Гэхдээ өмнө нэвтэрсэн бол асуудалгүй.")

# 2. СУРГАСАН ЗАГВАР БОЛОН ТОКЕНАЙЗЕР АЧААЛАХ
model_save_path = "./llama3-mongolian-combined"
print(f"Сургасан загварыг ачаалж байна: {model_save_path}")

try:
    # Токенайзер ачаалах
    tokenizer = AutoTokenizer.from_pretrained(model_save_path)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Загвар ачаалах
    model = AutoModelForCausalLM.from_pretrained(
        model_save_path,
        torch_dtype=torch.float32,
        device_map="auto",
    )
    print("Загвар болон токенайзер амжилттай ачаалагдлаа!")
except Exception as e:
    print(f"Загвар ачаалахад алдаа гарлаа: {e}")
    print("Анхны загварыг ашиглана...")

    # Хэрэв сургасан загвар ачаалах боломжгүй бол дахин оролдох
    print("Нээлттэй загвар ашиглаж байна...")
    model_name = "facebook/opt-350m"

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float32,
        device_map="auto",
    )

# 3. ӨРГӨТГӨСӨН АСУУЛТ-ХАРИУЛТЫН ӨГӨГДӨЛ
print("Өргөтгөсөн асуулт-хариултын өгөгдөл бэлдэж байна...")

qa_examples = [
    # Өмнөх өгөгдөл
    {"instruction": "Монгол улс хэдэн аймагтай вэ?", "input": "", "output": "Монгол улс нь 21 аймагтай. Эдгээр аймгууд нь Архангай, Баян-Өлгий, Баянхонгор, Булган, Говь-Алтай, Говьсүмбэр, Дархан-Уул, Дорноговь, Дорнод, Дундговь, Завхан, Орхон, Өвөрхангай, Өмнөговь, Сүхбаатар, Сэлэнгэ, Төв, Увс, Ховд, Хөвсгөл, Хэнтий."},
    {"instruction": "Монголын хамгийн өндөр уул юу вэ?", "input": "", "output": "Монголын хамгийн өндөр уул нь Хүйтэн уул буюу Таван Богд бөгөөд 4374 метр өндөр. Энэ уул нь Монголын баруун хойд хэсэгт Баян-Өлгий аймагт оршдог."},
    {"instruction": "Монгол хэлний цагаан толгой хэдэн үсэгтэй вэ?", "input": "", "output": "Монгол хэлний кирилл үсгийн цагаан толгой нь 35 үсэгтэй. Үүнд: А, Б, В, Г, Д, Е, Ё, Ж, З, И, Й, К, Л, М, Н, О, Ө, П, Р, С, Т, У, Ү, Ф, Х, Ц, Ч, Ш, Щ, Ъ, Ы, Ь, Э, Ю, Я."},
    {"instruction": "Монголын гол мөрний тухай хэлнэ үү", "input": "", "output": "Монгол улсын томоохон гол мөрөнд Сэлэнгэ, Орхон, Туул, Хэрлэн, Онон, Халхын гол, Ховд, Завхан, Идэр, Буянт зэрэг гол багтдаг. Сэлэнгэ мөрөн нь Монголын хамгийн том гол бөгөөд Орхон голын хамт урсгалынхаа 90 хувийг ОХУ-ын Эрхүү мужид орших Байгал нуурт юүлдэг."},
    {"instruction": "Монголын үндэсний цэцэг юу вэ?", "input": "", "output": "Монголын үндэсний цэцэг нь Алтан гагнуур (Латинаар Scabiosa comosa) юм. Алтан гагнуур нь Монголын тал хээрт ургадаг, шар өнгийн цэцэгтэй, эмийн шинж чанартай ургамал юм."},

    # Нэмэлт өгөгдөл - тоон мэдээлэл
    {"instruction": "Монгол улсын хүн ам хэд вэ?", "input": "", "output": "Монгол улсын хүн ам 2023 оны байдлаар ойролцоогоор 3.4 сая. Үүний тал орчим хувь нь нийслэл Улаанбаатар хотод амьдардаг."},
    {"instruction": "Монгол улсын газар нутгийн хэмжээ хэд вэ?", "input": "", "output": "Монгол улс нь 1,564,116 хавтгай дөрвөлжин километр газар нутагтай бөгөөд дэлхийн 18 дахь том улс юм."},
    {"instruction": "Монгол улсын нийслэл хот хэзээ байгуулагдсан бэ?", "input": "", "output": "Улаанбаатар хот нь 1639 онд Өргөө хэмээх нэртэйгээр анх байгуулагдсан. 1706 онд Их Хүрээ, 1911 онд Нийслэл Хүрээ, 1924 оноос Улаанбаатар нэртэй болсон."},

    # Нэмэлт өгөгдөл - түүх, соёл, ахуй
    {"instruction": "Монголын эзэнт гүрэн хэзээ байгуулагдсан бэ?", "input": "", "output": "Монголын эзэнт гүрэн нь 1206 онд Чингис хаан Монголын бүх овгуудыг нэгтгэж байгуулсан. Дараа нь 13, 14-р зууны үед энэ нь дэлхийн хамгийн том эзэнт гүрэн болж өргөжсөн."},
    {"instruction": "Монголчуудын уламжлалт сар шинийн баяр хэзээ тэмдэглэдэг вэ?", "input": "", "output": "Монголчууд уламжлалт сар шинийн баярыг цагаан сарын шинийн нэгэнд буюу хаврын тэргүүн сарын эхний өдөр тэмдэглэдэг. Цагаан сар нь сарын хуанлид суурилдаг тул Григорийн тооллын хувьд өөр өөр өдрүүдэд (ихэвчлэн 1-3 сард) тохиодог."},
    {"instruction": "Монголчуудын уламжлалт хувцас ямар вэ?", "input": "", "output": "Монголчуудын уламжлалт хувцас нь дээл юм. Дээл нь урт, өргөн, бүсээр чангалдаг урт ханцуйтай, өвөл дулаан байлгахын тулд зузаан хөвөнтэй юм. Үндэсний хувцасны бусад хэсгүүдэд малгай, гутал, бүс зэрэг орно. Дээлийг улирлаас хамаараад материал, зузаан, хээ угалзыг нь өөр өөрөөр урладаг."},

    # Нэмэлт өгөгдөл - ёс заншил, хоол, байгаль
    {"instruction": "Монголчуудын уламжлалт хоол хүнс юу вэ?", "input": "", "output": "Монголчуудын уламжлалт хоол хүнсэнд мах болон сүүн бүтээгдэхүүн чухал байр суурь эзэлдэг. Үүнд хуурсан мах, боодог, хорхог, цуйван, бууз, хуушуур зэрэг махан хоол, мөн айраг, тараг, аарц, ааруул, ээзгий зэрэг сүүн бүтээгдэхүүн багтдаг."},
    {"instruction": "Монголын үндэсний спорт юу вэ?", "input": "", "output": "Монголын үндэсний спортод бөх, сур харваа, морин уралдаан багтдаг бөгөөд эдгээр нь \"гурван эрийн наадам\"-ын бүрэлдэхүүн хэсэг юм. Мөн шагай харваа, шатар гэх мэт үндэсний тоглоомууд байдаг."},
    {"instruction": "Монголын хамгийн том нуур аль вэ?", "input": "", "output": "Монголын хамгийн том нуур бол Увс нуур юм. Увс нуур нь 3350 км² талбайтай бөгөөд Монгол-Оросын хилийн дагуу орших давст нуур юм. Энэ нь Төв Азийн хамгийн том нуур, дэлхийн хамгийн том давст нуурнуудын нэг юм."},

    # Нэмэлт өгөгдөл - улс төр, эдийн засаг
    {"instruction": "Монгол улсын төрийн тогтолцоо ямар вэ?", "input": "", "output": "Монгол улс нь парламентын засаглалтай бүгд найрамдах улс юм. Улсын дээд хууль тогтоох байгууллага нь Улсын Их Хурал (УИХ) бөгөөд 76 гишүүнтэй. Монгол улсын ерөнхийлөгчийг 6 жилийн хугацаатай сонгодог."},
    {"instruction": "Монгол улсын эдийн засгийн гол салбарууд юу вэ?", "input": "", "output": "Монгол улсын эдийн засгийн гол салбаруудад уул уурхай, хөдөө аж ахуй, мөн аялал жуулчлал орно. Монгол улс нүүрс, зэс, алт, уран зэрэг байгалийн баялаг ихтэй бөгөөд сүүлийн жилүүдэд уул уурхайн салбар эдийн засагт чухал байр суурь эзэлдэг болсон."},
    {"instruction": "Монгол улс хэзээ тусгаар тогтнолоо зарласан бэ?", "input": "", "output": "Монгол улс 1911 оны 12-р сарын 29-нд Манж Чин улсаас тусгаар тогтнолоо зарласан. Гэвч олон улсын хүлээн зөвшөөрөгдсөн бүрэн эрхт тусгаар тогтнолыг 1921 оны хувьсгалын дараа 1924 онд анх удаа олж авсан юм."},
]

# 4. ӨГӨГДЛИЙН САН ҮҮСГЭХ
qa_dataset = Dataset.from_dict({
    "instruction": [ex["instruction"] for ex in qa_examples],
    "input": [ex["input"] for ex in qa_examples],
    "output": [ex["output"] for ex in qa_examples]
})

print(f"Асуулт-хариултын өгөгдлийн сангийн хэмжээ: {len(qa_dataset)} бичлэг")

# 5. ӨГӨГДЛИЙГ БОЛОВСРУУЛАХ - АСУУЛТ-ХАРИУЛТЫН ХЭЛБЭРТ ОРУУЛАХ
# Асуулт-хариултын өгөгдлийг загварт зориулж форматлах
def prepare_qa_dataset(examples):
    texts = []
    for instruction, inp, output in zip(examples["instruction"], examples["input"], examples["output"]):
        if inp and inp != "":
            text = f"<|user|>\n{instruction}\n{inp}\n<|assistant|>\n{output}\n<|endoftext|>"
        else:
            text = f"<|user|>\n{instruction}\n<|assistant|>\n{output}\n<|endoftext|>"
        texts.append(text)
    return {"text": texts}

# Өгөгдлийг форматлах
processed_qa = qa_dataset.map(
    prepare_qa_dataset,
    batched=True,
    remove_columns=qa_dataset.column_names,
)

# 6. СУРГАЛТ/ТЕСТ ӨГӨГДӨЛД ХУВААХ
qa_train_test = processed_qa.train_test_split(test_size=0.1, seed=42)
print(f"Сургалтын өгөгдөл: {len(qa_train_test['train'])} бичлэг")
print(f"Тестийн өгөгдөл: {len(qa_train_test['test'])} бичлэг")

# 7. ТОКЕНЖУУЛАХ
def tokenize_function(examples):
    max_length = 256  # QA өгөгдөлд илүү урт хэмжээ

    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=max_length,
        return_tensors=None,
    )

# QA өгөгдлийг токенжуулах
print("Өгөгдлийг токенжуулж байна...")
tokenized_qa_dataset = qa_train_test.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

# 8. СУРГАЛТЫН ТОХИРГОО
print("Сургалтын тохиргоо үүсгэж байна...")
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# QA сургалтын аргументууд
qa_training_args = TrainingArguments(
    output_dir="./llama3-mongolian-qa",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    logging_dir="./logs-qa",
    logging_steps=5,
    save_steps=50,
    save_total_limit=1,
    learning_rate=1e-5,
    weight_decay=0.01,
    fp16=False,
    gradient_accumulation_steps=16,
    num_train_epochs=10,
    warmup_steps=20,
    report_to="tensorboard",
)

# 9. TRAINER ҮҮСГЭХ
print("Trainer объект үүсгэж байна...")
qa_trainer = Trainer(
    model=model,
    args=qa_training_args,
    train_dataset=tokenized_qa_dataset["train"],
    eval_dataset=tokenized_qa_dataset["test"],
    data_collator=data_collator,
)

# 10. СУРГАЛТЫГ ЭХЛҮҮЛЭХ
print("\nСургалтыг эхлүүлэхэд бэлэн байна!")
print("Сургалтыг дараах мөрийг ажиллуулан эхлүүлнэ:")
print("qa_trainer.train()")

# 11. ЗАГВАРЫГ ХАДГАЛАХ
print("\nСургалт дууссаны дараа загварыг хадгалахын тулд дараах мөрийг ажиллуулна:")
print('qa_model_path = "./llama3-mongolian-qa-finetuned"')
print('qa_trainer.save_model(qa_model_path)')
print('tokenizer.save_pretrained(qa_model_path)')

# 12. ЗАГВАРЫГ ТУРШИЖ ҮЗЭХ ФУНКЦ
def test_qa_model(question):
    """QA загварыг туршиж үзэх функц"""
    # Chat template ашиглах эсэхийг шалгах
    if hasattr(tokenizer, "apply_chat_template"):
        messages = [{"role": "user", "content": question}]
        inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to(model.device)
    else:
        # Fallback - chat template байхгүй бол
        prompt = f"<|user|>\n{question}\n<|assistant|>\n"
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Хариултыг үүсгэх
    outputs = model.generate(
        inputs.input_ids,
        max_new_tokens=150,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    # Үр дүнг буцаах
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # <|assistant|> хэсгийг ялгаж авах
    if "<|assistant|>" in generated_text:
        return generated_text.split("<|assistant|>\n")[1].split("<|endoftext|>")[0]
    return generated_text

print("\nСургалт дууссаны дараа туршилт хийхийн тулд дараах мөрүүдийг ажиллуулна:")
print('test_questions = [')
print('    "Монгол улс хэдэн аймагтай вэ?",')
print('    "Монголын хамгийн өндөр уул юу вэ?",')
print('    "Монголын үндэсний хувцас юу вэ?",')
print(']')
print('for question in test_questions:')
print('    print(f"Асуулт: {question}")')
print('    response = test_qa_model(question)')
print('    print(f"Хариулт: {response}\n")')

In [ ]:
qa_trainer.train()

In [ ]:
qa_model_path = "./llama3-mongolian-qa-finetuned"
qa_trainer.save_model(qa_model_path)
tokenizer.save_pretrained(qa_model_path)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Загварын зам
model_path = "./llama3-mongolian-qa-finetuned"

# 1. ЗАГВАР БОЛОН ТОКЕНАЙЗЕР АЧААЛАХ
print(f"Загварыг ачаалж байна: {model_path}")

# Токенайзер ачаалах
tokenizer = AutoTokenizer.from_pretrained(model_path)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Загвар ачаалах
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float32,
    device_map="auto",
)
print("Загвар болон токенайзер амжилттай ачаалагдлаа!")

# 2. ШИНЭЧИЛСЭН ТЕСТ ФУНКЦ
def test_qa_model_simple(question):
    """Шинэчилсэн QA загварыг туршиж үзэх функц"""

    # Шууд prompt форматлах
    prompt = f"<|user|>\n{question}\n<|assistant|>\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Хариултыг үүсгэх
    outputs = model.generate(
        inputs.input_ids,
        max_length=200,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    # Үр дүнг буцаах
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Бүгдийг буцаах
    return generated_text

# 3. ТЕСТИЙН АСУУЛТУУД
print("\nАсуулт-хариултын загварыг туршиж үзэж байна...")
test_questions = [
    "Монгол улс хэдэн аймагтай вэ?",
    "Монголын хамгийн өндөр уул юу вэ?",
    "Монголын үндэсний хувцас юу вэ?",
    "Монголын үндэсний спорт юу вэ?",
    "Монгол улсын хүн ам хэд вэ?",
    "Монголын үндэсний цэцэг юу вэ?",
]

for question in test_questions:
    print(f"\nАсуулт: {question}")
    response = test_qa_model_simple(question)
    print(f"Хариулт: {response}")

# 4. ДАРААГИЙН АЛХАМ
print("\n4-р даалгавар: Бусад чадварыг сургах руу шилжихэд бэлэн.")

In [ ]:
import os
from huggingface_hub import login
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

# 1. HUGGING FACE-Д НЭВТРЭХ (ХЭРЭВ ШААРДЛАГАТАЙ)
try:
    login(token="")
    print("Hugging Face-д нэвтэрлээ")
except Exception as e:
    print(f"Нэвтрэх үед алдаа гарлаа: {e}")
    print("Гэхдээ өмнө нэвтэрсэн бол асуудалгүй.")

# 2. QA ЗАГВАР АЧААЛАХ
print("QA загварыг ачаалж байна...")
model_path = "./llama3-mongolian-qa-finetuned"  # Сургасан QA загварын зам

try:
    # Токенайзер ачаалах
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Загвар ачаалах
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.float32,
        device_map="auto",
    )
    print("QA загвар амжилттай ачаалагдлаа!")
except Exception as e:
    print(f"QA загвар ачаалахад алдаа гарлаа: {e}")
    print("Илүү энгийн загвар ашиглана...")

    # Хэрэв QA загвар ачаалагдаагүй бол нээлттэй загвар ачаалах
    model_name = "facebook/opt-350m"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float32,
        device_map="auto",
    )

# 3. ОРЧУУЛГЫН ӨГӨГДӨЛ БЭЛДЭХ
print("Орчуулгын өгөгдөл бэлдэж байна...")

# Англи-Монгол орчуулгын өгөгдөл
translation_examples = [
    # Энгийн өгүүлбэрүүд
    {"english": "Hello, how are you?", "mongolian": "Сайн байна уу, та яаж байна?"},
    {"english": "My name is John.", "mongolian": "Миний нэр бол Жон."},
    {"english": "I am from Mongolia.", "mongolian": "Би Монголоос ирсэн."},
    {"english": "What time is it now?", "mongolian": "Одоо хэдэн цаг вэ?"},
    {"english": "Today is a beautiful day.", "mongolian": "Өнөөдөр маш сайхан өдөр байна."},
    {"english": "I like to read books.", "mongolian": "Би ном унших дуртай."},
    {"english": "Where is the nearest restaurant?", "mongolian": "Хамгийн ойрын ресторан хаана байна?"},
    {"english": "How much does this cost?", "mongolian": "Энэ хэд вэ?"},
    {"english": "The weather is very hot today.", "mongolian": "Өнөөдөр цаг агаар маш халуун байна."},
    {"english": "I need to go to the airport.", "mongolian": "Би нисэх онгоцны буудал руу явах хэрэгтэй."},

    # Түүх болон соёлын холбоотой
    {"english": "Mongolia has a rich history.", "mongolian": "Монгол улс баялаг түүхтэй."},
    {"english": "Chinggis Khan founded the Mongol Empire in 1206.", "mongolian": "Чингис хаан 1206 онд Монголын эзэнт гүрнийг үүсгэн байгуулсан."},
    {"english": "The Mongolian traditional clothing is called deel.", "mongolian": "Монголчуудын уламжлалт хувцсыг дээл гэдэг."},
    {"english": "Mongolian nomads live in gers (yurts).", "mongolian": "Монголын нүүдэлчид гэрт амьдардаг."},
    {"english": "Horse racing is very popular in Mongolia.", "mongolian": "Морин уралдаан Монголд маш түгээмэл."},

    # Байгаль, газарзүйн холбоотой
    {"english": "The Gobi Desert is located in southern Mongolia.", "mongolian": "Говь цөл нь Монголын өмнөд хэсэгт оршдог."},
    {"english": "Ulaanbaatar is the capital of Mongolia.", "mongolian": "Улаанбаатар нь Монгол улсын нийслэл."},
    {"english": "Lake Khuvsgul is one of the largest freshwater lakes in Asia.", "mongolian": "Хөвсгөл нуур нь Азийн хамгийн том цэнгэг усны нууруудын нэг юм."},
    {"english": "Mongolia has cold winters and hot summers.", "mongolian": "Монгол улсад өвөл хүйтэн, зун халуун байдаг."},
    {"english": "The highest mountain in Mongolia is Khuiten Peak.", "mongolian": "Монголын хамгийн өндөр уул нь Хүйтэн оргил."},

    # Өдөр тутмын ярианы өгүүлбэрүүд
    {"english": "Can you help me, please?", "mongolian": "Надад тусалж чадах уу?"},
    {"english": "I don't understand Mongolian.", "mongolian": "Би монгол хэл ойлгохгүй байна."},
    {"english": "I am learning Mongolian language.", "mongolian": "Би монгол хэл сурч байна."},
    {"english": "Where can I buy tickets?", "mongolian": "Би хаанаас тасалбар авч болох вэ?"},
    {"english": "How far is the city center?", "mongolian": "Хотын төв хэр хол вэ?"},

    # Хоол, хүнс, соёлын холбоотой
    {"english": "Mongolian cuisine includes a lot of dairy products.", "mongolian": "Монгол хоолонд сүүн бүтээгдэхүүн их ордог."},
    {"english": "Buuz is a traditional Mongolian food.", "mongolian": "Бууз бол Монголын уламжлалт хоол юм."},
    {"english": "The Naadam festival is celebrated every summer.", "mongolian": "Наадам баяр жил бүрийн зун тэмдэглэгддэг."},
    {"english": "Mongolian throat singing is called Khoomei.", "mongolian": "Монгол хөөмий дуу нь хөөмэй гэж нэрлэгддэг."},
    {"english": "Traditional Mongolian tea is made with milk, salt and butter.", "mongolian": "Монголын уламжлалт цай нь сүү, давс, шар тос зэргээр хийгддэг."},
]

# ӨГӨГДЛИЙН САН ҮҮСГЭХ
translation_dataset = Dataset.from_dict({
    "english": [ex["english"] for ex in translation_examples],
    "mongolian": [ex["mongolian"] for ex in translation_examples]
})

print(f"Орчуулгын өгөгдлийн сангийн хэмжээ: {len(translation_dataset)} бичлэг")

# 4. ӨГӨГДЛИЙГ БОЛОВСРУУЛАХ
# Орчуулгын өгөгдлийг сургалтын загварт зориулж форматлах
def prepare_translation_dataset(examples):
    texts = []
    for english, mongolian in zip(examples["english"], examples["mongolian"]):
        # Орчуулгын дасгалын формат: Англи → Монгол
        text = f"<|user|>\nTranslate to Mongolian: {english}\n<|assistant|>\n{mongolian}\n<|endoftext|>"
        texts.append(text)
    return {"text": texts}

# Өгөгдлийг форматлах
processed_translation = translation_dataset.map(
    prepare_translation_dataset,
    batched=True,
    remove_columns=translation_dataset.column_names,
)

# 5. СУРГАЛТ/ТЕСТ ӨГӨГДӨЛД ХУВААХ
translation_train_test = processed_translation.train_test_split(test_size=0.1, seed=42)
print(f"Сургалтын өгөгдөл: {len(translation_train_test['train'])} бичлэг")
print(f"Тестийн өгөгдөл: {len(translation_train_test['test'])} бичлэг")

# 6. ТОКЕНЖУУЛАХ
def tokenize_function(examples):
    max_length = 128  # Орчуулгын өгүүлбэрүүд харьцангуй богино

    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=max_length,
        return_tensors=None,
    )

# Өгөгдлийг токенжуулах
print("Өгөгдлийг токенжуулж байна...")
tokenized_translation_dataset = translation_train_test.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

# 7. СУРГАЛТЫН ТОХИРГОО
print("Сургалтын тохиргоо үүсгэж байна...")
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Орчуулгын сургалтын аргументууд
translation_training_args = TrainingArguments(
    output_dir="./llama3-mongolian-translation",
    per_device_train_batch_size=2,  # Бага өгөгдөл учир batch size нэмэгдүүлэв
    per_device_eval_batch_size=2,
    logging_dir="./logs-translation",
    logging_steps=5,
    save_steps=20,
    save_total_limit=1,
    learning_rate=2e-5,  # Орчуулгад зориулж суралцах хурдыг нэмэгдүүлэв
    weight_decay=0.01,
    fp16=False,
    gradient_accumulation_steps=8,
    num_train_epochs=50,  # Орчуулга сургахад илүү олон эргэлт
    warmup_steps=10,
    report_to="tensorboard",
)

# 8. TRAINER ҮҮСГЭХ
print("Trainer объект үүсгэж байна...")
translation_trainer = Trainer(
    model=model,
    args=translation_training_args,
    train_dataset=tokenized_translation_dataset["train"],
    eval_dataset=tokenized_translation_dataset["test"],
    data_collator=data_collator,
)

# 9. СУРГАЛТЫГ ЭХЛҮҮЛЭХ
print("\nСургалтыг эхлүүлэхэд бэлэн байна!")
print("Сургалтыг дараах мөрийг ажиллуулан эхлүүлнэ:")
print("translation_trainer.train()")

# 10. ЗАГВАРЫГ ХАДГАЛАХ
print("\nСургалт дууссаны дараа загварыг хадгалахын тулд дараах мөрийг ажиллуулна:")
print('translation_model_path = "./llama3-mongolian-translation-finetuned"')
print('translation_trainer.save_model(translation_model_path)')
print('tokenizer.save_pretrained(translation_model_path)')

# 11. ОРЧУУЛГА ТУРШИЖ ҮЗЭХ ФУНКЦ
def translate_to_mongolian(english_text):
    """Англи текстийг монгол хэл рүү орчуулах"""
    prompt = f"<|user|>\nTranslate to Mongolian: {english_text}\n<|assistant|>\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        inputs.input_ids,
        max_length=150,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    # Үр дүнг буцаах
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Орчуулсан хэсгийг ялгаж авах
    if "<|assistant|>" in generated_text:
        return generated_text.split("<|assistant|>\n")[1].split("<|endoftext|>")[0]
    return generated_text

print("\nСургалт дууссаны дараа туршилт хийхийн тулд дараах мөрүүдийг ажиллуулна:")
print('test_sentences = [')
print('    "Hello, nice to meet you!",')
print('    "Mongolia is a beautiful country.",')
print('    "I want to learn Mongolian language.",')
print('    "The weather is cold today.",')
print('    "Can you help me find the museum?",')
print(']')
print('for sentence in test_sentences:')
print('    print(f"Англи: {sentence}")')
print('    translation = translate_to_mongolian(sentence)')
print('    print(f"Монгол: {translation}\n")')

In [ ]:
translation_trainer.train()

In [ ]:
translation_model_path = "./llama3-mongolian-translation-finetuned"
translation_trainer.save_model(translation_model_path)
tokenizer.save_pretrained(translation_model_path)

In [ ]:
test_sentences = [
    "Hello, nice to meet you!",
    "Mongolia is a beautiful country.",
    "I want to learn Mongolian language.",
    "The weather is cold today.",
    "Can you help me find the museum?",
]
for sentence in test_sentences:
    print(f"Англи: {sentence}")
    translation = translate_to_mongolian(sentence)
    print(f"Монгол: {translation}\n")